# Goal setting

In [ ]:
import json

with open('gatchina.json', 'r') as file:
    analysis = json.load(file)

## Store

In [2]:
from src import embedding
import pandas as pd
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

store = InMemoryVectorStore(embedding=embedding)
indicators_df = pd.read_excel('./data/indicators.xlsx')[['Раздел', 'Код показателя', 'Название показателя', 'Единица измерения']].copy()

docs = []
for _,row in indicators_df.iterrows():
    data = row.to_dict()
    page_content = data['Название показателя']
    del data['Название показателя']
    doc = Document(page_content=page_content, metadata=data)
    docs.append(doc)

store.add_documents(docs)

['0a07b808-3efc-4dd3-a5ba-d3f2cc1a0ba5',
 '156e644d-054e-4463-aa42-ac1c957146a2',
 '2089c163-ef36-48bc-ac4f-e7134d7e7451',
 'd6429afd-7e0b-478d-8801-19411af1a258',
 'de96cb7c-d917-4746-9c3a-96743b8eec13',
 '5c706be9-b6fa-4e78-88f7-6b784e30a9ad',
 'a61011d6-26ea-4c2d-aee3-bdaf05b9ec06',
 'f0f94d0f-30ab-46c5-853c-7584beb23267',
 '8734f0ed-6a50-4e83-9350-5038a9e12da9',
 'f71d7c6d-8b22-472b-b9a8-aa982bdca322',
 'e5aeb719-fe39-499c-8cc3-2c1e939fa71d',
 '455e28da-189e-4e5f-bb16-a073ec231e35',
 '52a9ec43-bbab-435d-bba5-a99edae820e7',
 '646ed922-dc5d-4040-8dd9-af24e5d8c24f',
 'c9bdd11a-56ce-417d-95e6-8780d2bf75a2',
 '827d1073-4638-4f05-abd7-9c4b93eb27fe',
 '45c2dc56-1236-4840-9427-5e026715e0a7',
 'ed2c4d35-719f-4c96-aa82-b6d444286dd4',
 'a53144ec-49a7-4243-b5ea-b66663a63bf5',
 'b428e6ce-a18e-4955-8744-7466173bf902',
 '59453b26-f9f2-4398-85d9-501454750dda',
 'dd4166e4-c498-4baf-96fe-10898dd6c83a',
 'e49b0a7f-3d6a-4676-8083-1c3442c9d6ef',
 'ef9771bf-3666-4c5d-8116-4c1eb1f168de',
 '31d9efe4-0ea4-

In [3]:
from langchain.tools import tool

@tool
def search_tool(query : str):
    """
    Поиск по базе показателей
    """
    return store.similarity_search(query, k=10)

## Agents

In [4]:
from src import Agent
from src.tools import ddgs_tool, lightrag_tool
from pydantic import BaseModel, Field
from typing import Literal

class Indicator(BaseModel):
    code : str = Field(description='Код показателя')
    name : str = Field(description='Название показателя')
    target : Literal['min', 'max'] = Field(description='Направление целевого значения (минимизация или максимизация)')

class Task(BaseModel):
    description : str = Field(description='Описание задачи')
    indicators : list[Indicator] = Field(description='Целевые показатели')

class Goal(BaseModel):
    description : str = Field(description='Описание цели')
    tasks : list[Task] = Field(description='Задачи пространственного развития')

class ActorResponse(BaseModel):
    external_factors : list[str] = Field(description='Внешние факторы развития')
    internal_factors : list[str] = Field(description='Внутренние факторы развития')
    mission : str = Field(description='Миссия МО')
    goals : list[Goal] = Field(description='Цели пространственного развития')

ACTOR_PROMPT = """
На основе результатов аналитического этапа сформулируй:
- внешние и внутренние факторы развития;
- миссию МО;
- цели и задачи пространственного развития МО;
- систему целевых показателей.
---
ПРАВИЛА РАБОТЫ:
- Используй инструменты для поиска доступных показателей.
- Не придумывай показатели. Бери их из списка.
- Для каждого показателя укажи примерное целевое значение.
---
РЕЗУЛЬТАТЫ АНАЛИТИЧЕСКОГО ЭТАПА:
{analysis}
"""

actor = Agent(system_prompt=ACTOR_PROMPT.format(analysis=str(analysis)), tools=[search_tool], response_format=ActorResponse, is_checkpointer=True)

In [5]:
CRITIC_PROMPT = """
Ты строгий критик.
Сопоставь сформированное целеполагание с результатами аналитического этапа и предоставь обоснованную критику.
Будь краток.
---
РЕЗУЛЬТАТЫ АНАЛИТИЧЕСКОГО ЭТАПА:
{analysis}
"""

critic = Agent(system_prompt=CRITIC_PROMPT.format(analysis=str(analysis)), is_checkpointer=True)

In [6]:
from tqdm import tqdm

messages = []

max_iter = 3
for i in tqdm(range(max_iter)):

    agents = [actor]
    if (i+1) < max_iter:
        agents.append(critic)

    for agent in agents:
        messages = [str(m) for m in messages[-1:]]
        message = agent.invoke(messages)
        messages.append(message)

100%|██████████| 3/3 [03:36<00:00, 72.19s/it]


In [7]:
messages[-1].model_dump()

{'external_factors': ['Статус Гатчины как административного центра Ленинградской области с планируемым переездом высших органов власти',
  'Развитие транспортной инфраструктуры (строительство КАД-2, ремонт трассы Стрельна — Кипень — Гатчина)',
  "Реализация национальных проектов 'Жилье и городская среда', 'Безопасные качественные дороги'",
  "Участие в программе 'Умный город' как городского центра агломерации",
  'Рост спроса на жилье и коммерческую недвижимость в связи с изменением административного статуса',
  'Повышение качества рынка труда Ленинградской области (7-е место в РФ по итогам 2024 года)',
  "Развитие системы ООПТ Ленинградской области (55 объектов, включая 'Дивенский обрыв' на территории округа)"],
 'internal_factors': ['Высокий уровень автомобилизации (300 автомобилей на 1000 жителей) при протяженности дорог 1135 км',
  'Наличие развитой промышленной структуры (города Гатчина и Коммунар)',
  'Наличие железнодорожного сообщения с Санкт-Петербургом (16 пригородных поездов